In [69]:
import os 
import pandas as pd 
import numpy as np

In [ ]:
def ler_csv(arquivo_csv, cod_estacao, dt_inicio, dt_fim, encoding="utf-8"):
    """Lê o csv."""
    if arquivo_csv.endswith('.csv'):
        df = pd.read_csv(arquivo_csv, encoding=encoding, engine='python')
    else:
        df = pd.read_csv(arquivo_csv, sep='\t', encoding=encoding, engine='python')
    # Corrigir nomes das colunas para remover aspas e facilitar o acesso
    df.columns = [col.strip().replace('"', '') for col in df.columns]
    
    # Normalizar o nome da coluna de código da estação
    if "Cod.estacao" in df.columns:
        df.rename(columns={"Cod.estacao": "cod_estacao"}, inplace=True)
    if "Cod_estacao" in df.columns:
        df.rename(columns={"Cod_estacao": "cod_estacao"}, inplace=True)
    
    # Converter a coluna Data para datetime
    if "Data" in df.columns:
        df["Data"] = pd.to_datetime(df["Data"], errors='coerce')
    else:
        raise ValueError('Coluna "Data" não encontrada no arquivo CSV')
    
    # Filtrar pelos parâmetros
    df = df[df["cod_estacao"] == cod_estacao]
    df = df[(df["Data"] >= pd.to_datetime(dt_inicio)) & (df["Data"] <= pd.to_datetime(dt_fim))]
    return df

def criar_dataframe_estacao(
    caminho_estacao: str,
    tipo_dado: str,
    lista_estacoes: list,
    dt_inicio: str = '1966-01-01',
    dt_fim: str = '2023-12-31',
    encoding: str = None
) -> 'pd.DataFrame':
    """
    Cria um DataFrame consolidado com dados de várias estações.

    Args:
        caminho_estacao (str): Caminho para os arquivos das estações.
        tipo_dado (str): Coluna de interesse (ex: 'chuva').
        lista_estacoes (list): Lista de identificadores de estações.
        dt_inicio (str): Data inicial no formato 'YYYY-MM-DD'.
        dt_fim (str): Data final no formato 'YYYY-MM-DD'.
        encoding (str, optional): Encoding do arquivo CSV.

    Returns:
        pd.DataFrame: DataFrame com coluna 'Data' e colunas para cada estação.
    """
    df = ler_csv(caminho_estacao, lista_estacoes[0], dt_inicio, dt_fim, encoding=encoding)
    df_chuva = df[['Data',tipo_dado]].rename(columns={tipo_dado:str(lista_estacoes[0])})
    for estacao in lista_estacoes[1:]:  
        df = ler_csv(caminho_estacao, estacao, dt_inicio, dt_fim, encoding=encoding)
        df_chuva = df_chuva.merge(df[['Data',tipo_dado]].rename(columns={tipo_dado:str(estacao)}), on='Data', how='outer')
    return df_chuva

In [ ]:
arquivo_plu = os.path.join(os.getcwd(), "PLU_Series_ANA.txt")
arquivo_flu = os.path.join(os.getcwd(), "FLU_Series_ANA.txt")

In [ ]:
# # DADOS DE CHUVAS
# # 1547002
# df_1547002 = ler_csv(arquivo_plu, 1547002, '1974-01-01', '2012-12-31')

# # 1547011
# df_1547011 = ler_csv(arquivo_plu, 1547011, '1971-01-01', '2009-12-31')

# # 1547071
# df_1547071 = ler_csv(arquivo_plu, 1547071, '2010-01-01', '2021-12-31')

# # 1547072
# df_1547072 = ler_csv(arquivo_plu, 1547071, '2010-01-01', '2021-12-31')

# # 1547073
# df_1547073 = ler_csv(arquivo_plu, 1547073, '2010-01-01', '2016-12-31')

# # 1547078
# df_1547078 = ler_csv(arquivo_plu, 1547078, '2008-01-01', '2023-12-31')


In [ ]:
# DADOS DE CHUVA
lista_estacoes_plu = [ 1547002, 1547011, 1547071, 1547072, 1547073, 1547078
                    ]

df_chuva = criar_dataframe_estacao(arquivo_plu, 'Chuva', lista_estacoes_plu, dt_inicio='1966-01-01', dt_fim='2023-12-31')

In [ ]:
# DADOS DE VAZÃO 
# 60473000
df_60473000 = ler_csv(arquivo_flu, 60473000, '1971-01-01', '2023-12-31')

# 60471185
df_60471185 = ler_csv(arquivo_flu, 60471185, '2010-01-01', '2017-12-31')

# 60474000
df_60474000 = ler_csv(arquivo_flu, 60474000, '1979-01-01', '1994-12-31')

# 60471200
df_60471200 = ler_csv(arquivo_flu, 60471200, '1990-01-01', '2023-12-31')

# 60474100
df_60474100 = ler_csv(arquivo_flu, 60474100, '1995-01-01', '2023-12-31')

# 60476000
df_60476000 = ler_csv(arquivo_flu, 60476000, '1971-01-01', '1975-12-31')

# 60476150
df_60476150 = ler_csv(arquivo_flu, 60476150, '1993-01-01', '1998-12-31')

# 60476100
df_60476100 = ler_csv(arquivo_flu, 60476100, '1978-01-01', '2014-12-31')

In [ ]:
# Vazão ANA

lista_estacoes_flu = [60473000, 60471185, 60474000, 60471200, 60474100, 60476000, 60476150, 60476100]

df_vazao = criar_dataframe_estacao(arquivo_flu, 'Vazao', lista_estacoes_flu, dt_inicio='1966-01-01', dt_fim='2023-12-31')

In [ ]:
# df_vazao = df_60473000[['Data','Vazao']].rename(columns={'Vazao':'60473000'})
# df_vazao = df_vazao.merge(df_60471185[['Data','Vazao']].rename(columns={'Vazao':'60471185'}), on='Data', how='outer')
# df_vazao = df_vazao.merge(df_60474000[['Data','Vazao']].rename(columns={'Vazao':'60474000'}), on='Data', how='outer')
# df_vazao = df_vazao.merge(df_60471200[['Data','Vazao']].rename(columns={'Vazao':'60471200'}), on='Data', how='outer')
# df_vazao = df_vazao.merge(df_60474100[['Data','Vazao']].rename(columns={'Vazao':'60474100'}), on='Data', how='outer')
# df_vazao = df_vazao.merge(df_60476000[['Data','Vazao']].rename(columns={'Vazao':'60476000'}), on='Data', how='outer')
# df_vazao = df_vazao.merge(df_60476150[['Data','Vazao']].rename(columns={'Vazao':'60476150'}), on='Data', how='outer')
# df_vazao = df_vazao.merge(df_60476100[['Data','Vazao']].rename(columns={'Vazao':'60476100'}), on='Data', how='outer')
# df_vazao = df_vazao.sort_values(by='Data').reset_index(drop=True)


In [ ]:
# df_chuva = df_1547011[['Data','Chuva']].rename(columns={'Chuva':'1547011'})
# df_chuva = df_chuva.merge(df_1547002[['Data','Chuva']].rename(columns={'Chuva':'1547002'}), on='Data', how='outer')
# df_chuva = df_chuva.merge(df_1547071[['Data','Chuva']].rename(columns={'Chuva':'1547071'}), on='Data', how='outer')
# df_chuva = df_chuva.merge(df_1547072[['Data','Chuva']].rename(columns={'Chuva':'1547072'}), on='Data', how='outer')
# df_chuva = df_chuva.merge(df_1547073[['Data','Chuva']].rename(columns={'Chuva':'1547073'}), on='Data', how='outer')
# df_chuva = df_chuva.merge(df_1547078[['Data','Chuva']].rename(columns={'Chuva':'1547078'}), on='Data', how='outer')
# df_chuva = df_chuva.sort_values(by='Data').reset_index(drop=True)

In [ ]:
# Renomear colunas para evitar conflitos
# df_vazao.columns = ['Data'] + [f'{col}_ANA' for col in df_vazao.columns if col != 'Data']
# df_vazao_caesb.columns = ['Data'] + [f'{col}_CAESB' for col in df_vazao_caesb.columns if col != 'Data']

In [ ]:
df_chuva['Data'] = df_chuva['Data'].astype(str)
df_vazao['Data'] = df_vazao['Data'].astype(str)

df_base = df_chuva.merge(df_vazao, on='Data', how='outer') \
                  .merge(df_vazao, on='Data', how='outer')

NameError: name 'df_chuva' is not defined

In [ ]:
# Adiciona a interpretação do coeficiente de Pearson conforme a tabela fornecida
def interpretar_pearson(r):
    if pd.isnull(r):
        return 'Nula'
    r = abs(r)
    if r == 0:
        return 'Nula'
    elif 0 < r <= 0.20:
        return 'Ínfima fraca'
    elif 0.21 <= r <= 0.40:
        return 'Fraca'
    elif 0.41 <= r <= 0.60:
        return 'Moderada'
    elif 0.61 <= r <= 0.80:
        return 'Forte'
    elif 0.81 <= r <= 0.99:
        return 'Ínfima Forte'
    elif r == 1:
        return 'Perfeita'
    else:
        return 'Indefinida'     

# Loop para calcular a correlação de Pearson entre todas as estações pluviométricas e fluviométricas
plu_cols = [col for col in df_base.columns if str(col).startswith('15')]
flu_cols = [col for col in df_base.columns if str(col).startswith('60')]

correlacoes = []
for plu in plu_cols:
    for flu in flu_cols:
        corr = df_base[plu].corr(df_base[flu])
        correlacoes.append({'Pluviometrica': plu, 'Fluviometrica': flu, 'Pearson': corr})

df_correlacoes = pd.DataFrame(correlacoes)
df_correlacoes['Interpretacao'] = df_correlacoes['Pearson'].apply(interpretar_pearson)
df_correlacoes['Pearson'] = df_correlacoes['Pearson'].round(2)

In [83]:
# Transformar de longa para larga (pivotar a tabela)
df_correlacoes_larga_texto = df_correlacoes.pivot(index='Fluviometrica', columns='Pluviometrica', values='Interpretacao')
df_correlacoes_larga_texto

Pluviometrica,1547004,1547005,1547018,1547029,1548051
Fluviometrica,,,,,
60477300,Fraca,Nula,Fraca,Fraca,Moderada
60477600,Indefinida,Nula,Fraca,Indefinida,Moderada
60477700,Moderada,Nula,Moderada,Moderada,Nula


In [84]:
df_correlacoes_larga_valor = df_correlacoes.pivot(index='Fluviometrica', columns='Pluviometrica', values='Pearson')
df_correlacoes_larga_valor

Pluviometrica,1547004,1547005,1547018,1547029,1548051
Fluviometrica,,,,,
60477300,0.30,NaN,0.29,0.31,0.43
60477600,0.40,NaN,0.36,0.40,0.50
60477700,0.47,NaN,0.48,0.45,NaN
